In [ ]:
# 최신 패키지 설치 (셀에서 실행)
!pip install -U transformers accelerate peft bitsandbytes torchvision
# A100, H100 환경을 위한 Flash Attention 2 설치 (설치에 몇 분 소요될 수 있음)
!pip install flash-attn --no-build-isolation

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
from PIL import Image, ImageEnhance
from torchvision import transforms
import pandas as pd
from tqdm import tqdm
import bitsandbytes as bnb

# ==========================================
# 1. 하이퍼파라미터 및 설정 (최적화 반영)
# ==========================================
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
SEED = 42
IMAGE_SIZE = 640       # 화질 향상을 위해 384 -> 640 상향
NUM_EPOCHS = 3         # 학습 횟수 증가
BATCH_SIZE = 4         # A100 VRAM(40GB/80GB)에 맞춰 상향 가능 (OOM 발생 시 2로 조정)
GRAD_ACCUM = 4         # 실질적인 배치 사이즈 = BATCH_SIZE * GRAD_ACCUM
LEARNING_RATE = 2e-4   # LoRA 학습률

# ==========================================
# 2. 이미지 전처리 및 증강 파이프라인
# ==========================================
def enhance_image(image):
    """대비 및 선명도 향상"""
    image = ImageEnhance.Contrast(image).enhance(1.2)
    image = ImageEnhance.Sharpness(image).enhance(1.5)
    return image

train_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.RandomRotation(degrees=3), 
])

class CustomVQADataset(Dataset):
    def __init__(self, df, processor, is_train=True):
        self.df = df
        self.processor = processor
        self.is_train = is_train
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 데이터 로드
        image_path = self.df.iloc[idx]['image_path']
        image = Image.open(image_path).convert("RGB")
        
        # 전처리 및 증강 적용
        image = enhance_image(image)
        if self.is_train:
            image = train_transform(image)
            
        question = self.df.iloc[idx]['question']
        answer = self.df.iloc[idx]['answer']
        
        # Qwen2.5-VL 프롬프트 포맷팅
        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]}
        ]
        
        text_prompt = self.processor.apply_chat_template(messages, add_generation_prompt=True)
        # 정답(answer)을 이어서 학습하도록 구성
        text_prompt += answer + "<|im_end|>"
        
        inputs = self.processor(
            text=[text_prompt],
            images=[image],
            padding=True,
            return_tensors="pt"
        )
        
        return inputs

# ==========================================
# 3. 데이터 로딩 (샘플링 확대)
# ==========================================
# 실제 데이터 경로에 맞게 수정하세요.
train_df = pd.read_csv("train.csv") 
# 기존 200개에서 전체 데이터 사용 또는 대량 샘플링으로 변경
# 예: train_df = train_df.sample(n=3000, random_state=SEED).reset_index(drop=True)

processor = AutoProcessor.from_pretrained(MODEL_ID)
train_dataset = CustomVQADataset(train_df, processor, is_train=True)

# 메모리 최적화를 위한 커스텀 collate_fn 생략 시 DataLoader의 기본 배치 처리 사용
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# ==========================================
# 4. 모델 로드 및 LoRA 설정 (A100/H100 최적화)
# ==========================================
print("모델 로딩 중... (Flash Attention 2 및 Bfloat16 적용)")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,            # A100/H100을 위한 Bfloat16 적용
    attn_implementation="flash_attention_2", # 메모리 절약 및 속도 향상
    device_map="auto"
)

# 모델 Gradient Checkpointing 활성화 (메모리 절약)
model.gradient_checkpointing_enable()

# LoRA 설정 고도화
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ==========================================
# 5. 최적화 도구 및 학습 루프 (스케줄러 추가)
# ==========================================
# VRAM을 아끼기 위해 8-bit AdamW 옵티마이저 사용
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=LEARNING_RATE)

# 안정적인 수렴을 위한 Cosine Learning Rate Scheduler 추가
total_steps = len(train_dataloader) * NUM_EPOCHS // GRAD_ACCUM
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1), # 초기 10%는 워밍업
    num_training_steps=total_steps
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.train()

print("🚀 학습 시작!")
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for step, batch in enumerate(progress_bar):
        # 데이터를 모델 디바이스로 이동 (차원 축소 처리 포함)
        inputs = {k: v.squeeze(0).to(device) if v.dim() > 2 else v.to(device) for k, v in batch.items()}
        
        # 모델의 출력 계산 (라벨은 input_ids와 동일하게 주어 Next Token Prediction 수행)
        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss / GRAD_ACCUM
        
        loss.backward()
        
        # Gradient Accumulation 적용
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_dataloader):
            optimizer.step()
            scheduler.step() # 스케줄러 업데이트
            optimizer.zero_grad()
            
        total_loss += loss.item() * GRAD_ACCUM
        progress_bar.set_postfix(loss=loss.item() * GRAD_ACCUM)
        
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} Completed | Average Loss: {avg_loss:.4f}")

# 모델 저장
print("학습 완료! 모델을 저장합니다.")
model.save_pretrained("./qwen2.5-vl-vqa-finetuned")
processor.save_pretrained("./qwen2.5-vl-vqa-finetuned")